# RF-DETR training and test evaluation

This notebook reproduces the RF-DETR Nano and Medium workflow for the Urban Disaster Monitor dataset. It trains a selected variant and evaluates it on the **test split**.

The RF-DETR training API expects a COCO export with `train`, `valid`, and `test` folders containing `_annotations.coco.json`. Download the dataset in COCO format from [Roboflow Universe](https://universe.roboflow.com/ufrnprojects-xlut9/urban-disaster-monitor/dataset/4) before running the training cells.

The parameters below match the recorded runs: 50 epochs, batch size 4, gradient accumulation 4, and resolutions of 384 px (Nano) or 576 px (Medium).

## 1. Dependencies and GPU

Run this notebook in a Python 3.10+ environment with a CUDA GPU when possible.

In [ ]:
!pip install "rfdetr[train]>=1.8.3" pandas --quiet

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 2. Dataset validation

Set `DATASET_DIR` to the root directory of the COCO export. The validation checks prevent a training run against the repository's YOLO-format export by mistake.

In [ ]:
from pathlib import Path

DATASET_DIR = Path("/content/dataset")  # Update if your COCO export is elsewhere
required_annotations = [DATASET_DIR / split / "_annotations.coco.json" for split in ("train", "valid", "test")]
missing = [path for path in required_annotations if not path.exists()]
if missing:
    missing_paths = "\n".join(f"- {path}" for path in missing)
    raise FileNotFoundError(
        "A COCO export is required. Missing annotation files:\n" + missing_paths
    )

print(f"Using COCO dataset: {DATASET_DIR.resolve()}")

## 3. Configure and train

Choose `nano` or `medium`. `NUM_CLASSES = 7` preserves the class configuration recorded in the existing RF-DETR artifacts (`objects` plus the six project classes). Set `RUN_TEST = True` to produce test-split results at the end of training.

In [ ]:
from rfdetr import RFDETRMedium, RFDETRNano

VARIANT = "medium"  # "nano" or "medium"
NUM_CLASSES = 7
EPOCHS = 50
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 1e-4
RESOLUTION = {"nano": 384, "medium": 576}[VARIANT]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("runs") / f"rfdetr-{VARIANT}-urban-disaster"
RUN_TEST = True

model_class = {"nano": RFDETRNano, "medium": RFDETRMedium}[VARIANT]
model = model_class(num_classes=NUM_CLASSES)

model.train(
    dataset_dir=str(DATASET_DIR),
    output_dir=str(OUTPUT_DIR),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    lr=LEARNING_RATE,
    resolution=RESOLUTION,
    device=DEVICE,
    eval_interval=1,
    run_test=RUN_TEST,
)

## 4. Inspect recorded metrics

RF-DETR writes training configuration and metrics to the run directory. Keep these files with the checkpoint so results can be compared with the YOLOv26m test evaluation.

In [ ]:
import pandas as pd

metrics_path = OUTPUT_DIR / "metrics.csv"
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    metric_columns = [
        column for column in ("epoch", "val/mAP_50", "val/mAP_50_95", "val/precision", "val/recall", "val/F1")
        if column in metrics.columns
    ]
    display(metrics[metric_columns].dropna(how="all").tail())
else:
    print(f"Run training first; metrics not found at {metrics_path}")

for artifact in ("training_config.json", "results.json", "checkpoint_best_total.pth"):
    print(f"{artifact}: {(OUTPUT_DIR / artifact).exists()}")

## 5. Run inference with the best checkpoint

Use an image from the test split for a qualitative check. The trained checkpoint can also be uploaded to the project's Hugging Face repository for use by the Gradio app.

In [ ]:
image_extensions = {".jpg", ".jpeg", ".png", ".webp"}
test_images = sorted(path for path in (DATASET_DIR / "test").iterdir() if path.suffix.lower() in image_extensions)
if not test_images:
    raise FileNotFoundError("No test images were found beside test/_annotations.coco.json.")

checkpoint_path = OUTPUT_DIR / "checkpoint_best_total.pth"
if not checkpoint_path.exists():
    raise FileNotFoundError(f"Best checkpoint not found: {checkpoint_path}")

best_model = model_class(num_classes=NUM_CLASSES, pretrain_weights=str(checkpoint_path))
detections = best_model.predict(str(test_images[0]), threshold=0.30)
print(f"Test image: {test_images[0].name}")
print(f"Detections: {len(detections.xyxy)}")
detections